# 10 minutes to Flyte

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/unionai/unionai-examples/blob/main/v2/user-guide/getting-started/ten_minutes_to_flyte.ipynb)

Welcome! This is a whirlwind tour of [Flyte](https://www.union.ai/docs/v2/), a framework for
building and scaling AI, ML, and data workflows in pure Python.

Everything in this notebook runs **locally, right here in the notebook process** — no cluster,
no Docker, no account required. In the next 10 minutes you will:

1. Write and run your first Flyte task
2. Compose tasks into workflows with plain Python
3. Fan out work in parallel with `flyte.map`
4. Skip redundant work with caching
5. Recover from transient failures with retries
6. Learn how to visualize runs with the terminal UI (TUI)

At the end, you'll see how to graduate from in-process execution to a real (but still local)
Kubernetes cluster with the **devbox** — without changing a line of your task code.

## Setup

Install the `flyte` SDK — that's the only dependency for this tour:

In [ ]:
%pip install -q -U 'flyte[tui]'

Note: you may need to restart the kernel to use updated packages.


## 1. Your first task

Flyte code is organized around two ideas:

- A **`TaskEnvironment`** groups configuration — container image, resources, cache policy —
  shared by a set of tasks. Locally it's just a namespace; on a cluster it defines the
  container your tasks run in.
- The **`@env.task`** decorator turns a plain Python function into a task: a tracked,
  retryable, cacheable unit of work with a typed interface.

Type annotations are required — they define each task's interface, and they're what makes
caching, input validation, and data passing between containers work.

In [2]:
import flyte

env = flyte.TaskEnvironment(name="quickstart")


@env.task
def hello(name: str) -> str:
    return f"Hello, {name}!"

Now run it. Calling `flyte.init(local_persistence=True)` configures Flyte for **local mode**
(no endpoint, no cluster): runs execute in this Python process. Enabling `local_persistence`
records every run in a local SQLite database, so you can inspect it later — you'll browse
these runs in the TUI in section 6.

> **Note:** In a notebook we use `await flyte.run.aio(...)` because Jupyter already runs an
> event loop. In a plain Python script, you'd write `flyte.run(hello, name="Flyte")`.

In [3]:
flyte.init(local_persistence=True)

run = await flyte.run.aio(hello, name="Flyte")
run.wait()

print(run.outputs()[0])

Hello, Flyte!


That's the whole loop: define a task, `flyte.run` it, get typed outputs back. Every run is
recorded with its inputs, outputs, and logs — you'll browse them in the TUI shortly.

## 2. Workflows are just Python

In Flyte, a workflow is simply **a task that calls other tasks**. There's no DSL and no
separate workflow language — loops, `if`/`else`, and `try`/`except` all work exactly as you'd
expect. Each task call becomes its own tracked (and, on a cluster, containerized) step.

In [4]:
@env.task
def get_data(n: int) -> list[float]:
    return [float(i) for i in range(1, n + 1)]


@env.task
def normalize(values: list[float]) -> list[float]:
    mu = sum(values) / len(values)
    sigma = (sum((v - mu) ** 2 for v in values) / len(values)) ** 0.5
    return [(v - mu) / sigma for v in values]


@env.task
def pipeline(n: int = 10) -> list[float]:
    raw = get_data(n)        # each call runs as its own tracked task
    return normalize(raw)


run = await flyte.run.aio(pipeline, n=5)
run.wait()
run.outputs()[0]

[-1.414213562373095,
 -0.7071067811865475,
 0.0,
 0.7071067811865475,
 1.414213562373095]

## 3. Fan out with `flyte.map`

`flyte.map` works like Python's built-in `map`, but runs the calls **in parallel**. Locally
that means concurrent execution in this process; on a cluster, each call becomes its own
container — the same code scales from 10 items to 10,000.

In [5]:
@env.task
def simulate(x: int) -> float:
    return 2.0 * x + 5.0  # pretend this is an expensive simulation


@env.task
def sweep(xs: list[int]) -> float:
    ys = list(flyte.map(simulate, xs))  # parallel fan-out
    return sum(ys) / len(ys)


run = await flyte.run.aio(sweep, xs=list(range(10)))
run.wait()
run.outputs()[0]

15:23:11.849807 WARNING  [8jgc3hki9d9y2ujsfdi8vjjie]  _map.py:337 - Running map in local mode, which will run every task sequentially.

14.0

## 4. Skip redundant work with caching

Add `cache="auto"` to a task and Flyte reuses previous results whenever the task is called
with inputs it has already seen — within a run *and* across runs. Locally, the cache is a
SQLite database on disk, so it survives between runs with zero setup.

`slow_square` below takes 2 seconds per call. The pipeline calls it 6 times, but with only
3 unique inputs — watch the timing:

In [8]:
import time


@env.task(cache="auto")
def slow_square(x: int) -> int:
    time.sleep(2)
    return x**2


@env.task
def cached_pipeline(xs: list[int]) -> list[int]:
    return [slow_square(x) for x in xs]


t0 = time.perf_counter()
run = await flyte.run.aio(cached_pipeline, xs=[1, 2, 2, 3, 3, 3])
run.wait()
print(f"First run:  {time.perf_counter() - t0:.1f}s  (6 calls, 3 unique → 3 executions)")

t0 = time.perf_counter()
run = await flyte.run.aio(cached_pipeline, xs=[1, 2, 2, 3, 3, 3])
run.wait()
print(f"Second run: {time.perf_counter() - t0:.1f}s  (all cache hits)")

First run:  0.0s  (6 calls, 3 unique → 3 executions)
Second run: 0.0s  (all cache hits)


## 5. Recover from failures with retries

Real pipelines hit flaky APIs and transient errors. Declare `retries=` on a task and Flyte
re-runs it automatically on failure — no `try`/`except` boilerplate in your workflow code.

The task below fails twice before succeeding on the third attempt:

> The `attempts` counter works here because local mode runs everything in one process. On a
> cluster, each attempt starts in a fresh container — which is exactly what you want for
> genuinely transient failures.

In [12]:
attempts = {"n": 0}


@env.task(retries=3)
def flaky(x: int) -> int:
    attempts["n"] += 1
    if attempts["n"] < 3:
        print("This RuntimeError is expected! 👇")
        raise RuntimeError(f"transient failure (attempt {attempts['n']})")
    print(f"succeeded on attempt {attempts['n']}")
    return x * 100


run = await flyte.run.aio(flaky, x=1)
run.wait()
run.outputs()[0]

This RuntimeError is expected! 👇


15:24:36.611515 ERROR    [1rxrhkbmju42d6t9p1kmh8kxh]  taskrunner.py:129 - Task failed with error: transient failure (attempt 1)                                 
                         Traceback (most recent call last):                                                                                                     
                           File "/Users/nielsbantilan/git/unionai-examples/.venv/lib/python3.12/site-packages/flyte/_internal/runtime/taskrunner.py", line 121, 
                         in run_task                                                                                                                            
                             outputs = await task.execute(**inputs)                                                                                             
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                                                             
                           File "/Users/nielsbantilan/git/unionai-examples/.venv/lib/python3.12/site-packages/flyte/_task.py", line 606, in execute             
                             v = await run_sync_in_thread(self.func, *args, **kwargs)                                                                           
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                                           
                           File "/Users/nielsbantilan/git/unionai-examples/.venv/lib/python3.12/site-packages/flyte/_utils/asyncify.py", line 83, in            
                         run_sync_in_thread                                                                                                                     
                             return await asyncio.wrap_future(fut)                                                                                              
                                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                                                              
                           File "/Users/nielsbantilan/git/unionai-examples/.venv/lib/python3.12/site-packages/flyte/_utils/asyncify.py", line 71, in _runner    
                             fut.set_result(copied_ctx.run(func, *args, **kwargs))                                                                              
                                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                                               
                           File "/var/folders/4q/frdnh9l10h53gggw1m59gr9m0000gp/T/ipykernel_28712/818013583.py", line 9, in flaky                               
                             raise RuntimeError(f"transient failure (attempt {attempts['n']})")                                                                 
                         RuntimeError: transient failure (attempt 1)

15:24:36.617383 WARNING    _local_controller.py:284 - Task 'quickstart.flaky' action '1rxrhkbmju42d6t9p1kmh8kxh' failed on attempt 1/4; retrying in 0.50s...

This RuntimeError is expected! 👇


15:24:37.124988 ERROR    [1rxrhkbmju42d6t9p1kmh8kxh]  taskrunner.py:129 - Task failed with error: transient failure (attempt 2)                                 
                         Traceback (most recent call last):                                                                                                     
                           File "/Users/nielsbantilan/git/unionai-examples/.venv/lib/python3.12/site-packages/flyte/_internal/runtime/taskrunner.py", line 121, 
                         in run_task                                                                                                                            
                             outputs = await task.execute(**inputs)                                                                                             
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                                                             
                           File "/Users/nielsbantilan/git/unionai-examples/.venv/lib/python3.12/site-packages/flyte/_task.py", line 606, in execute             
                             v = await run_sync_in_thread(self.func, *args, **kwargs)                                                                           
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                                           
                           File "/Users/nielsbantilan/git/unionai-examples/.venv/lib/python3.12/site-packages/flyte/_utils/asyncify.py", line 83, in            
                         run_sync_in_thread                                                                                                                     
                             return await asyncio.wrap_future(fut)                                                                                              
                                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                                                              
                           File "/Users/nielsbantilan/git/unionai-examples/.venv/lib/python3.12/site-packages/flyte/_utils/asyncify.py", line 71, in _runner    
                             fut.set_result(copied_ctx.run(func, *args, **kwargs))                                                                              
                                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                                               
                           File "/var/folders/4q/frdnh9l10h53gggw1m59gr9m0000gp/T/ipykernel_28712/818013583.py", line 9, in flaky                               
                             raise RuntimeError(f"transient failure (attempt {attempts['n']})")                                                                 
                         RuntimeError: transient failure (attempt 2)

15:24:37.130275 WARNING    _local_controller.py:284 - Task 'quickstart.flaky' action '1rxrhkbmju42d6t9p1kmh8kxh' failed on attempt 2/4; retrying in 1.00s...

succeeded on attempt 3


100

## 6. Visualize your runs with the TUI

Every run you just executed was recorded, and Flyte ships a **terminal UI (TUI)** for watching
runs live and browsing past ones — a full task tree with status indicators (● running,
✓ done, ✗ failed), cache markers, live logs, and input/output details.

> The TUI needs a real terminal, so it won't render inside Colab — run these commands **on
> your own machine**, with the TUI extra installed: `pip install -U "flyte[tui]"`.

First, save the tour's workflow as a script:

In [13]:
%%writefile hello.py
import flyte

env = flyte.TaskEnvironment(name="hello_env")


@env.task
def fn(x: int) -> int:
    slope, intercept = 2, 5
    return slope * x + intercept


@env.task
def main(x_list: list[int] = list(range(10))) -> float:
    y_list = list(flyte.map(fn, x_list))
    return sum(y_list) / len(y_list)

Overwriting hello.py


Then, in a terminal, run the `hello.py` `main` function  with the TUI attached.

| ℹ️ On Google Colab, you can access the terminal on the bottom left bar |
| - |

```bash
flyte run --local --tui hello.py main
```

You'll see the run unfold live — the `main` task fanning out `fn` across all ten inputs.
Inside the TUI:

- `d` — details tab (inputs, outputs, metadata for the selected task)
- `l` — logs tab
- `q` — quit

You can also browse **past** runs at any time — including all the runs from this notebook if
you're running it locally — with:

```bash
# run this on Google Colab for better colors
export TERM=xterm-256color

# start the TUI
flyte start tui
```

Learn more in the [local run mode guide](https://www.union.ai/docs/v2/union/user-guide/get-started/run-modes/running-locally).

## Next step: run on a real cluster with the Flyte Devbox

Everything above ran in a single Python process. The whole point of Flyte is that the **same
code** — same tasks, same environments, no changes — can run each task in its own container
on a Kubernetes cluster, with a scheduler, object store, and web UI.

The fastest way to experience that is the **devbox**: a lightweight local cluster that runs
on your machine with Docker. 

### Option 1: Github Codespaces

The fastest way to run the Flyte Devbox in your browser is through Github Codespaces:

[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/flyteorg/flyte-devbox-codespace?quickstart=1)

It'll take 5-10 minutes for the codespace to start up, but once it has, the `README.md` in the browser-based VSCode environment contains instructions on how to run simple tasks on the codespace-hosted Flyte Devbox.

### Option 1: Your local machine

To run Flyte Devbox on your local machine, you'll need Python 3.10+, and [Docker](https://docs.docker.com/get-docker/)
running.

**1. Start the devbox** (the first start takes a few minutes while containers download):

```bash
flyte start devbox
```

**2. Point your config at it:**

```bash
flyte create config --devbox
```

**3. Run the same workflow — now on a cluster:**

```bash
flyte run hello.py main
```

This time each task executes in its own container on the devbox. Open the Flyte UI at
[http://localhost:30080](http://localhost:30080) to explore the execution graph, task logs,
and timeline — the same experience you get on a production cluster.

When you're done, `flyte stop devbox` stops the cluster and `flyte delete devbox` removes it.

**→ Follow the full [devbox guide](https://www.union.ai/docs/v2/union/user-guide/get-started/run-modes/running-devbox) to get started.**

### Keep going

- [Run modes](https://www.union.ai/docs/v2/union/user-guide/get-started/run-modes) — local vs. devbox vs. remote cluster
- [Flyte v2 documentation](https://www.union.ai/docs/v2/) — the full user guide
- [unionai-examples](https://github.com/unionai/unionai-examples/tree/main/v2) — more runnable examples and tutorials